In [ ]:
## Step 1: Check GPU Availability

# To detect GPU availability and print the GPU name.
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        details = tf.config.experimental.get_device_details(gpu)
        print(details.get("device_name", "Unknown GPU"))
else:
    print("No GPU detected")


In [ ]:
## Step 2: Set Global Policy for Mixed Precision Training

# To set the global policy to 'mixed_float16' for mixed precision training.
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

print("Global policy:", mixed_precision.global_policy())


In [ ]:
## Step 3: Read an Image with a Unicode Path using OpenCV

# To read an image with a Unicode path using OpenCV.
import numpy as np
import cv2
    
def read_image_unicode(path):
    try:
        data = np.fromfile(path, dtype=np.uint8)
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        return img
    except Exception as e:
        print("Failed to read:", path, e)
        return None

In [ ]:
## Step 4: Pre-process an Image using OpenCV

# To perform pre-processing on an image, including converting to grayscale, applying Gaussian blur, and performing Canny edge detection.
import cv2

def pre_processing(img):
    # convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # smooth (important for pencil)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)


    # Canny edge detection
    edges = cv2.Canny(
        gray,
        threshold1=60,
        threshold2=60
    )

    return edges

In [ ]:
# To load an image, perform pre-processing, and display the Canny edge image using Matplotlib.
import cv2
import matplotlib.pyplot as plt

# ---- load image ----
img_path = "./dataset\ha\{4F8984ED-4079-4D72-9464-66A14D5446AD}.png"
img = cv2.imread(img_path)

if img is None:
    raise ValueError("Image not found")

canny_img = pre_processing(img)

plt.figure(figsize=(4,4))
plt.title("Canny Edge Image")
plt.imshow(canny_img, cmap="gray")
plt.axis("off")

plt.show()


In [ ]:
## Step 5: Load a Dataset of Images from a Directory Structure

# To load a dataset of images from a directory structure, perform pre-processing on each image, and return the images and their corresponding labels as NumPy arrays.
import os
import numpy as np

DATASET_DIR = "./dataset"   # contains ka/, ga/, la/
IMG_SIZE = (128, 128)     # single letter → square is best

def load_dataset():
    X = []
    y = []

    total_files = 0
    read_ok = 0

    for label in sorted(os.listdir(DATASET_DIR)):
        class_dir = os.path.join(DATASET_DIR, label)
        if not os.path.isdir(class_dir):
            continue

        for fname in os.listdir(class_dir):
            if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
                continue

            total_files += 1
            img_path = os.path.join(class_dir, fname)

            
            img = read_image_unicode(img_path)

            if img is None:
                continue

            read_ok += 1

           

            img = pre_processing(img)
            img = cv2.resize(img, IMG_SIZE)
            img = np.stack([img, img, img], axis=-1)
            img = img / 255.0

            X.append(img)
            y.append(label)

    print("Total image files found:", total_files)
    print("Images read successfully:", read_ok)
    print("Images kept:", len(X))

    return np.array(X), np.array(y)


In [ ]:
## Step 6: Spliting the Dataset into (train, valid, test)

# 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42

X, y_text = load_dataset()
le = LabelEncoder()
y = le.fit_transform(y_text)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)